In [1]:
import pandas as pd
import polars as pl

In [ ]:
df  = pl.read_csv("complaints/complaints.csv")
df.head()

Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64
"""2025-02-26T21:46:29.000Z""","""Vehicle loan or lease""","""Loan""","""Struggling to pay your loan""","""Denied request to lower paymen…","""I am filing a formal complaint…",null,"""Van Horn Automotive Group, Inc…","""MI""","""499XX""",null,"""Web""","""2025-02-26T22:00:16.000Z""","""Closed with explanation""","""No""",12225144
"""2025-03-20T04:16:23.000Z""","""Debt collection""","""Auto debt""","""Communication tactics""","""You told them to stop contacti…","""I am filing this complaint due…",null,"""Van Horn Automotive Group, Inc…","""MI""","""499XX""",null,"""Web""","""2025-03-20T04:25:05.000Z""","""Closed with explanation""","""No""",12551969
"""2025-04-20T02:56:09.000Z""","""Student loan""","""Federal student loan servicing""","""Struggling to repay your loan""","""Problem with forgiveness, canc…","""MOHELA is illegally not proces…",null,"""MOHELA""","""VA""","""22203""",null,"""Web""","""2025-04-20T03:04:33.000Z""","""Closed with explanation""","""No""",13076106
"""2025-04-16T13:16:54.000Z""","""Student loan""","""Federal student loan servicing""","""Dealing with your lender or se…","""Trouble with how payments are …","""Date | Subject/Type | Details/…",null,"""MOHELA""","""DE""","""198XX""",null,"""Web""","""2025-04-16T13:25:05.000Z""","""Closed with explanation""","""No""",13016738
"""2025-04-16T20:39:22.000Z""","""Student loan""","""Federal student loan servicing""","""Dealing with your lender or se…","""Trouble with how payments are …","""I had submitted an Income driv…",null,"""MOHELA""","""CA""","""94124""",null,"""Web""","""2025-04-16T20:54:46.000Z""","""Closed with explanation""","""No""",13015269


In [3]:
df.shape

(15712797, 16)

In [22]:
for i in df['Product'].unique():
    print(i)

Credit card
Credit reporting or other personal consumer reports
Bank account or service
Debt or credit management
Money transfer, virtual currency, or money service
Virtual currency
Mortgage
Checking or savings account
Payday loan, title loan, personal loan, or advance loan
Payday loan, title loan, or personal loan
Credit card or prepaid card
Debt collection
Credit reporting
Student loan
Other financial service
Payday loan
Money transfers
Prepaid card
Consumer Loan
Vehicle loan or lease
Credit reporting, credit repair services, or other personal consumer reports


In [23]:
# Banking products only
BANKING_PRODUCTS = [
    "Student loan",
    "Credit card",
    "Bank account or service",
    "Mortgage",
    "Debt collection"
]

# Filter dataset
filtered_df = (
    df
    .filter(
        pl.col("Product").is_in(BANKING_PRODUCTS)
    )
    .filter(
        pl.col("Consumer complaint narrative").is_not_null()
    )
    .with_columns(
        pl.col("Consumer complaint narrative")
        .str.len_chars()
        .alias("n_chars")
    )
    .filter(
        pl.col("n_chars") >= 100
    )
)

print(filtered_df.shape)

(747899, 17)


In [24]:
filtered_df = filtered_df.with_columns([
    pl.col("Sub-product")
        .fill_null("Not specified"),

    pl.col("Sub-issue")
        .fill_null("Not specified")
])

In [25]:
filtered_df = filtered_df.select([
    "Consumer complaint narrative",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue"
])

In [26]:
TARGET_PER_PRODUCT = 2000

sampled_df = (
    filtered_df
    .group_by("Product")
    .map_groups(
        lambda group:
        group.sample(
            n=min(len(group), TARGET_PER_PRODUCT),
            seed=42
        )
    )
)

In [27]:
sampled_df = (
    filtered_df
    .group_by(["Product", "Issue"])
    .map_groups(
        lambda group:
        group.sample(
            n=min(len(group), 200),
            seed=42
        )
    )
)

In [28]:
print(
    sampled_df
    .group_by("Product")
    .len()
    .sort("len", descending=True)
)

print(
    sampled_df
    .group_by("Issue")
    .len()
    .sort("len", descending=True)
)

shape: (5, 2)
┌─────────────────────────┬──────┐
│ Product                 ┆ len  │
│ ---                     ┆ ---  │
│ str                     ┆ u32  │
╞═════════════════════════╪══════╡
│ Credit card             ┆ 7304 │
│ Mortgage                ┆ 2803 │
│ Student loan            ┆ 2280 │
│ Debt collection         ┆ 2200 │
│ Bank account or service ┆ 1000 │
└─────────────────────────┴──────┘
shape: (78, 2)
┌─────────────────────────────────┬─────┐
│ Issue                           ┆ len │
│ ---                             ┆ --- │
│ str                             ┆ u32 │
╞═════════════════════════════════╪═════╡
│ Improper use of your report     ┆ 600 │
│ Problem with a company's inves… ┆ 600 │
│ Incorrect information on your … ┆ 600 │
│ Credit monitoring or identity … ┆ 460 │
│ Problem with a credit reportin… ┆ 400 │
│ …                               ┆ …   │
│ Balance transfer fee            ┆ 51  │
│ Cash advance                    ┆ 43  │
│ Cash advance fee                ┆ 41  

In [ ]:
sampled_df.write_csv(
    "Data/cfpb_banking_curated.csv"
)

In [18]:
import pandas as pd 
import polars as pl
import json

In [ ]:
sampled_df = pl.read_csv("Data/cfpb_banking_curated.csv")
sampled_df.head()

Consumer complaint narrative,Product,Sub-product,Issue,Sub-issue
str,str,str,str,str
"""Navy Federal Credit Union has …","""Mortgage""","""Conventional home mortgage""","""Problem with a credit reportin…","""Investigation took more than 3…"
"""On XX/XX/XXXX I emailed the Cu…","""Mortgage""","""Conventional home mortgage""","""Problem with a credit reportin…","""Investigation took more than 3…"
"""Loancare has failed to investi…","""Mortgage""","""FHA mortgage""","""Problem with a credit reportin…","""Their investigation did not fi…"
"""My mortgage payments were on a…","""Mortgage""","""Conventional home mortgage""","""Problem with a credit reportin…","""Investigation took more than 3…"
"""I am writing to bring to your …","""Mortgage""","""VA mortgage""","""Problem with a credit reportin…","""Their investigation did not fi…"


In [20]:
sampled_df['Product'].unique()

Product
str
"""Debt collection"""
"""Credit card"""
"""Student loan"""
"""Mortgage"""
"""Bank account or service"""


In [21]:
import json
# Fill missing labels
sampled_df = sampled_df.with_columns([
    pl.col("Sub-product").fill_null("Not specified"),
    pl.col("Sub-issue").fill_null("Not specified")
])

SYSTEM_PROMPT = """
You are a banking complaint intake assistant.

Analyze the complaint narrative and identify:

1. Product
2. Sub-product
3. Issue
4. Sub-issue

Return JSON only.
"""

USER_TEMPLATE = """
Analyze the following customer complaint and determine the appropriate complaint classification.

Customer Complaint:
{complaint}
"""

samples = []

for row in sampled_df.iter_rows(named=True):

    assistant_output = {
        "product": row["Product"],
        "sub_product": row["Sub-product"],
        "issue": row["Issue"],
        "sub_issue": row["Sub-issue"]
    }

    sample = {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": USER_TEMPLATE.format(
        complaint=row["Consumer complaint narrative"]
    )
            },
            {
                "role": "assistant",
                "content": json.dumps(
                    assistant_output,
                    ensure_ascii=False
                )
            }
        ]
    }

    samples.append(sample)

In [22]:
from sklearn.model_selection import train_test_split

pdf = sampled_df.to_pandas()

In [15]:
train_df, temp_df = train_test_split(
    pdf,
    test_size=0.2,
    random_state=42,
    stratify=pdf["Product"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["Product"]
)

In [24]:
def create_sample(row):

    assistant_output = {
        "product": row["Product"],
        "sub_product": row["Sub-product"],
        "issue": row["Issue"],
        "sub_issue": row["Sub-issue"]
    }

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": USER_TEMPLATE.format(
        complaint=row["Consumer complaint narrative"]
    )
            },
            {
                "role": "assistant",
                "content": json.dumps(assistant_output)
            }
        ]
    }


def save_jsonl(df, path):

    with open(path, "w", encoding="utf-8") as f:

        for _, row in df.iterrows():

            sample = create_sample(row)

            f.write(
                json.dumps(sample, ensure_ascii=False)
                + "\n"
            )

In [ ]:
save_jsonl(train_df, "JSON_CHAT_TEMPLATE/train.jsonl")
save_jsonl(val_df, "JSON_CHAT_TEMPLATE/validation.jsonl")
save_jsonl(test_df, "JSON_CHAT_TEMPLATE/test.jsonl")